In [2]:
import os
import re
import math
import json
from tqdm import tqdm
import random
from dotenv import load_dotenv
from huggingface_hub import login
import matplotlib.pyplot as plt
import numpy as np
import pickle
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import chromadb
from testing import Tester
from items import Item

In [15]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
hf_token = os.getenv('HF_TOKEN')
pinecone_api_key = os.getenv('PINECONE_API_KEY')
login(hf_token)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
openai = OpenAI()

In [10]:
with open('test.pkl', 'rb') as file:
    test = pickle.load(file)
print(test[0].prompt)
test = random.sample(test,len(test))
print(test[0].prompt)

How much does this cost to the nearest dollar?

OEM AC Compressor w/A/C Repair Kit For Ford F150 F-150 V8 & Lincoln Mark LT 2007 2008 - BuyAutoParts NEW
As one of the world's largest automotive parts suppliers, our parts are trusted every day by mechanics and vehicle owners worldwide. This A/C Compressor and Components Kit is manufactured and tested to the strictest OE standards for unparalleled performance. Built for trouble-free ownership and 100% visually inspected and quality tested, this A/C Compressor and Components Kit is backed by our 100% satisfaction guarantee. Guaranteed Exact Fit for easy installation 100% BRAND NEW, premium ISO/TS 16949 quality - tested to meet or exceed OEM specifications Engineered for superior durability, backed by industry-leading unlimited-mileage warranty Included in this K

Price is $374.00
How much does this cost to the nearest dollar?

Ruvati Satin Brass Matte Gold Stainless Steel 60/40 Double Bowl Apron-Front Farmhouse Kitchen Sink -
BRASS TONE M

In [11]:
test[0]

<Ruvati 33-inch Satin Brass Matte Gold Stainless Steel 60/40 Double Bowl Apron-Front Farmhouse Kitchen Sink - RVH9742GG = $599.0>

In [12]:
def make_context(similars, prices):
    message = "To provide some context, here are some other items that might be similar to the item you need to estimate.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

In [13]:
def messages_for(item, similars, prices):
    system_message = "You estimate prices of items. Reply only with the price, no explanation"
    user_prompt = make_context(similars, prices)
    user_prompt += "And now the question for you:\n\n"
    user_prompt += item.test_prompt().replace(" to the nearest dollar","").replace("\n\nPrice is $","")
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": "Price is $"}
    ]

In [14]:
index_name = "products-vectorstore"

In [16]:
from pinecone import Pinecone
pc = Pinecone(
    api_key=pinecone_api_key
)
index = pc.Index(index_name)

In [17]:
def description(item):
    text = item.prompt.replace("How much does this cost to the nearest dollar?\n\n", "")
    return text.split("\n\nPrice is $")[0]

In [19]:
print(description(test[0]))

Ruvati Satin Brass Matte Gold Stainless Steel 60/40 Double Bowl Apron-Front Farmhouse Kitchen Sink -
BRASS TONE MATTE GOLD finish - Gold or Brass colored hardware is now a very trendy kitchen style. Now you can add a brass-tone sink to match your fixtures. Sink is constructed of thick 16 GAUGE T-304 Grade Stainless Steel that will NEVER RUST or tarnish. The COLORED FINISH is applied as an outer layer using an innovative nano-PVD technique that makes it INCREDIBLY TOUGH and RESISTANT TO WEARING or PEELING. MATTE texture adds to the beauty of the finish, and RESISTS FINGERPRINTS or stains. | Heavy duty SoundGuard undercoating and thick rubber NoiseDefend padding


In [20]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [34]:
def vector(item):
    return np.array(model.encode([description(item)])).astype(float).tolist()

In [32]:
vector(test[0])

(1, 384)

In [39]:
description(test[0])

'Ruvati Satin Brass Matte Gold Stainless Steel 60/40 Double Bowl Apron-Front Farmhouse Kitchen Sink -\nBRASS TONE MATTE GOLD finish - Gold or Brass colored hardware is now a very trendy kitchen style. Now you can add a brass-tone sink to match your fixtures. Sink is constructed of thick 16 GAUGE T-304 Grade Stainless Steel that will NEVER RUST or tarnish. The COLORED FINISH is applied as an outer layer using an innovative nano-PVD technique that makes it INCREDIBLY TOUGH and RESISTANT TO WEARING or PEELING. MATTE texture adds to the beauty of the finish, and RESISTS FINGERPRINTS or stains. | Heavy duty SoundGuard undercoating and thick rubber NoiseDefend padding'

In [38]:
[i['metadata']['description'] for i in index.query(top_k=5,vector=vector(test[0]),include_metadata=True)['matches']]

['Ruvati 30 x 20 inch Drop-in Topmount Granite Composite Kitchen Sink Single Bowl - Silver Gray -\nMade in ITALY | Natural stone finish. Color SILVER GRAY | Constructed of composite GRANITE - 80% crushed natural granite for extreme durability and strength. The sink is colored all the way through, for a rich and consistent look that will NEVER DISCOLOR | Proprietary treatment makes the surface resistant to stains | Can withstand high TEMPERATURES up to 650 °F DROP-IN TOPMOUNT installation. Cutout templates and mounting clips are provided for both type of installation | Pre-drilled with 3 partially-drilled knockout holes (standard 1-3/8 size) for faucet and soap dispenser installation (you can open up as many holes as',
 'Elkay Lustertone Classic Single Bowl Drop-In Stainless Steel ADA Sink\nSink Dimensions 22 X 19-1/2 X 6 Min. Cabinet Size 27 ADA COMPLIANT Product is ADA compliant when properly installed. REPAIRABLE FINISH Finish is scratch resistant to heavy-duty use. Deep scratches ar

In [40]:
def find_similars(item):
    results = index.query(vector=vector(item), top_k=5, include_metadata=True)['matches']
    documents = [m['metadata']['description'] for m in results]
    prices = [m['metadata']['price'] for m in results]
    return documents, prices

In [41]:
print(test[1].prompt)

How much does this cost to the nearest dollar?

COMOPHOTO Gold Purple Birthday Backdrop 7x5ft Adult Women Golden High Heels Balloon Champagne Diamond Photography Background for Party Banner Decorations Photo Booth Photoshoot Supplies
COMOPHOTO Gold Purple Birthday Backdrop Adult Women Golden High Heels Balloon Champagne Diamond Photography Background for Party Banner Decorations Photo Booth Photoshoot Supplies Thin to hang up straightly or picture is clear,realistic,Stereo sense is perfect for any professional or private photography. Please note screens are different in color correction,subtle color difference is the same time,because of our background material or pictorial ink in different batches,the background when printed will have very small backdrops will send by folded. Here are ways to remove it up tightly with a cylinder for 3-4 days,it will be necessary

Price is $18.00


In [42]:
documents, prices = find_similars(test[1])

In [43]:
print(make_context(documents, prices))

To provide some context, here are some other items that might be similar to the item you need to estimate.

Potentially related product:
ABLIN 7x5ft Lemon Theme Birthday Backdrop for Girls Our Little Lemonade is Turning One White Pink Stripes Photography Background Baby Girl 1st Birthday Party Decorations Props
Package Content： Only one photography background (Folded ). Background Stands and clips not included. About Creases In order to facilitate transport and packaging, this photography background will be folded to send, It will lead to the creases (If you mind the folds please carefully buy). But do not worry, it will not affect the use, recovery method as follows 1.Use the iron, gently ironing from the back, pressing the back,to avoid potential steam damage, place a layer of dry cloth between the backdrop and the iron will be a nice measure. 2.Roll it up tightly with a cylinder,
Price is $16.99

Potentially related product:
Music Night Backdrop Blue Photography Backgrounds Starligh

In [44]:
print(messages_for(test[1], documents, prices))

[{'role': 'system', 'content': 'You estimate prices of items. Reply only with the price, no explanation'}, {'role': 'user', 'content': "To provide some context, here are some other items that might be similar to the item you need to estimate.\n\nPotentially related product:\nABLIN 7x5ft Lemon Theme Birthday Backdrop for Girls Our Little Lemonade is Turning One White Pink Stripes Photography Background Baby Girl 1st Birthday Party Decorations Props\nPackage Content： Only one photography background (Folded ). Background Stands and clips not included. About Creases In order to facilitate transport and packaging, this photography background will be folded to send, It will lead to the creases (If you mind the folds please carefully buy). But do not worry, it will not affect the use, recovery method as follows 1.Use the iron, gently ironing from the back, pressing the back,to avoid potential steam damage, place a layer of dry cloth between the backdrop and the iron will be a nice measure. 2.

In [45]:
def get_price(s):
    s = s.replace('$','').replace(',','')
    match = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(match.group()) if match else 0

In [46]:
get_price("The price for this is $99.99")

99.99

In [47]:
def gpt_4o_mini_rag(item):
    documents, prices = find_similars(item)
    response = openai.chat.completions.create(
        model="gpt-4o-mini", 
        messages=messages_for(item, documents, prices),
        seed=42,
        max_tokens=5
    )
    reply = response.choices[0].message.content
    return get_price(reply)

In [53]:
gpt_4o_mini_rag(test[3])

221.36

In [52]:
test[3].price

231.62

In [54]:
Tester.test(gpt_4o_mini_rag, test)

1: Guess: $1,299.00 Truth: $599.00 Error: $700.00 SLE: 0.60 Item: Ruvati 33-inch Satin Brass Matte Gold St...
2: Guess: $17.99 Truth: $17.99 Error: $0.00 SLE: 0.00 Item: COMOPHOTO Gold Purple Birthday Backdrop ...
3: Guess: $89.99 Truth: $168.48 Error: $78.49 SLE: 0.39 Item: TRULUX HTL-UWW-5MKIT Indoor High-Output ...
4: Guess: $221.36 Truth: $231.62 Error: $10.26 SLE: 0.00 Item: ICON 01725 Tandem Axle Fender Skirt FS17...
5: Guess: $382.45 Truth: $336.21 Error: $46.24 SLE: 0.02 Item: PowerSport Front Rear Brakes and Rotors ...
6: Guess: $199.99 Truth: $330.48 Error: $130.49 SLE: 0.25 Item: Olideauto Smart Home Electric Sliding Do...
7: Guess: $249.99 Truth: $465.81 Error: $215.82 SLE: 0.39 Item: Bandai DX Chogokin Movie Ver. VF-1S Valk...
8: Guess: $199.00 Truth: $172.61 Error: $26.39 SLE: 0.02 Item: Baldwin Hardware 5225.031.ENTR Knob Set
9: Guess: $209.99 Truth: $206.69 Error: $3.30 SLE: 0.00 Item: RuffStuff Specialties Diff Cover For Dan...
10: Guess: $499.00 Truth: $511.50 Error: 

KeyboardInterrupt: 

In [55]:
from agents.openai_agent import OpenAIAgent

In [56]:
import logging
root = logging.getLogger()
root.setLevel(logging.INFO)

In [57]:
agent = OpenAIAgent(index)

INFO:root:[Openai Agent] Intializing OpenAI Agent
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:root:[Openai Agent] Openai Agent is ready


In [58]:
agent.price("Quadcast HyperX condenser mic for high quality podcasting")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Openai Agent] Openai Ai Agent is going to predict the price with context of 5 similar prodoct
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 502 Bad Gateway"
INFO:openai._base_client:Retrying request to /chat/completions in 0.403907 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Openai Agent] Openai Agent completed - predicting $139.99


139.99

In [59]:
from agents.pricer_agent import PricerAgent

In [60]:
agent2 = PricerAgent()

INFO:root:[Pricer Agent] Pricer Agent initialized - connecting to modal
INFO:root:[Pricer Agent] Pricer Agent is ready


In [61]:
agent2.price("Quadcast HyperX condenser mic for high quality podcasting")

INFO:root:[Pricer Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Pricer Agent] Received price: 130.0


130.0